# ChemBreak | Gemma 4 locally hosted judge

This notebook runs the reconstructed ChemBreak package. **RHS=1–5**, **OFS=1–10**, success threshold **OFS=10 (provisional)**. Both independent cumulative scores come from **the locally served Gemma 4 26B A4B IT model**, not GPT. Python validates evidence excerpts against response offsets and logs cryptographic hashes. Evaluations are separate and errors remain unresolved.

**Run a local vLLM server before LIVE experiments.** The package provides `scripts/start_gemma_server.sh` and a benign `scripts/probe_local_judge.py` smoke test. The model must be hosted in a separate process with sufficient GPU memory. The default official BF16 model requires approximately 58 GB just for loading weights, so running it alongside a target model on an A100 80 GB can be infeasible. Use a second GPU/host with a localhost tunnel or independently validate an appropriately quantized checkpoint. Do not install modern vLLM into the legacy target-model Python environment without dependency compatibility testing.

Default `DRY_RUN=True` uses harmless mocks and does not start or require Gemma. Use a unique run ID when switching judge, scale or weights. Do not merge 1–5 and 1–10 OFS observations.


In [ ]:
# Cell 2: experiment controls
DRY_RUN = True                  # Switch to False only after local Gemma server is running
RUN_ID = None                   # New independent run ID for the new evaluator/scoring version
SEED = 34034
TASK_LIMIT = None
PROBE_TOKENIZERS = False
PROBE_ROLES = False             # Live benign smoke test for all three roles (calls local judge)
SHOW_LIVE_TEXT = True
LIVE_TEXT_LIMIT = 1500

# Use ZIP created from this rebuilt release. Upload it to the Colab Files panel,
# or accept the notebook's upload prompt. Do not fetch the old GitHub main branch.
SOURCE_PACKAGE = 'uploaded_zip'   # 'uploaded_zip' or 'github'
PACKAGE_ZIP_PATH = '/content/ChemBreak_Gemma4_Rebuilt.zip'
REPOSITORY_URL = 'https://github.com/Jollychuks/ChemBreak.git'
REPOSITORY_BRANCH = 'main'
PROJECT_SUBDIR = 'chembreak'
GEMMA_JUDGE_MODEL = 'google/gemma-4-26B-A4B-it'
JUDGE_BASE_URL = 'http://127.0.0.1:8000/v1'


In [ ]:
# Hugging Face cache fix: configure before importing transformers / huggingface_hub.
# Some Transformers helper calls use the default global cache even when cache_dir is passed.
import os
from pathlib import Path
HF_CACHE_HOME = Path('/content/chembreak_storage/cache/huggingface').resolve()
HF_CACHE_HUB = HF_CACHE_HOME / 'hub'
HF_CACHE_HUB.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE_HOME)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_HUB)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_CACHE_HUB)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_HUB)
# Prove that the configured cache is writable before downloading model shards.
probe = HF_CACHE_HUB / '.chembreak_cache_write_probe'
probe.write_text('ok', encoding='utf-8')
probe.unlink()
print('Hugging Face cache is writable:', HF_CACHE_HUB)


In [ ]:
# Cell 3: load this rebuilt package from the uploaded ZIP (default), not stale GitHub code
from pathlib import Path
import subprocess, zipfile, shutil
REPO_ROOT = Path('/content/chembreak_repo')
if SOURCE_PACKAGE == 'uploaded_zip':
    source_zip = Path(PACKAGE_ZIP_PATH)
    if not source_zip.is_file():
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('Upload ChemBreak_Gemma4_Rebuilt.zip')
        source_zip = Path('/content') / next(iter(uploaded))
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    REPO_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source_zip) as archive:
        # Refuse paths escaping the extraction root.
        for member in archive.infolist():
            target = (REPO_ROOT / member.filename).resolve()
            if not target.is_relative_to(REPO_ROOT.resolve()):
                raise ValueError('Unsafe ZIP entry')
        archive.extractall(REPO_ROOT)
else:
    if not (REPO_ROOT / '.git').is_dir():
        subprocess.run(['git','clone','--depth','1','--branch',REPOSITORY_BRANCH,
                        REPOSITORY_URL,str(REPO_ROOT)],check=True)
    else:
        subprocess.run(['git','-C',str(REPO_ROOT),'pull','--ff-only'],check=True)
PROJECT_ROOT = (REPO_ROOT / PROJECT_SUBDIR).resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError(f'ChemBreak package not found: {PROJECT_ROOT}')
print('ChemBreak source:',PROJECT_ROOT)


In [ ]:
# Cell 4: install dependencies and activate this independent package
import subprocess, sys, importlib, site

def run_install(args):
    print('RUNNING:', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), check=True)

run_install([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
requirements = PROJECT_ROOT / 'requirements-cloud-ml.txt'
if not requirements.exists():
    raise FileNotFoundError(f'Missing dependency file: {requirements}')
run_install([sys.executable, '-m', 'pip', 'install', '-r', requirements])
run_install([sys.executable, '-m', 'pip', 'install', '-e', PROJECT_ROOT])
# Editable installs update a .pth file that a running kernel may not reload.
# Activate the local src tree explicitly so this cell works without a runtime restart.
SRC_ROOT = (PROJECT_ROOT / 'src').resolve()
PACKAGE_ROOT = SRC_ROOT / 'chembreak'
if not PACKAGE_ROOT.exists():
    raise FileNotFoundError(f'ChemBreak source package not found: {PACKAGE_ROOT}')
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
site.addsitedir(str(SRC_ROOT))
for module_name in list(sys.modules):
    if module_name == 'chembreak' or module_name.startswith('chembreak.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
import chembreak
loaded_from = Path(chembreak.__file__).resolve()
if PACKAGE_ROOT not in loaded_from.parents:
    raise RuntimeError(f'ChemBreak imported from the wrong location: {loaded_from}')
print('ChemBreak import OK:', chembreak.__version__, loaded_from)


In [ ]:
# Cell 5: GPU and persistent working storage
from pathlib import Path
import torch

STORAGE_ROOT = Path('/content/chembreak_storage').resolve()
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
print('STORAGE_ROOT =', STORAGE_ROOT)
print('CUDA available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU =', torch.cuda.get_device_name(0))
    print('bfloat16 supported =', torch.cuda.is_bf16_supported() if hasattr(torch.cuda, 'is_bf16_supported') else 'unknown')
elif not DRY_RUN:
    raise RuntimeError('A live ChemBreak target run requires a CUDA GPU runtime.')


In [ ]:
# Cell 6: Vertex credentials only. The Gemma judge is served locally, with NO GPT key.
import os
if not DRY_RUN:
    if not os.environ.get('GOOGLE_CLOUD_PROJECT', '').strip():
        os.environ['GOOGLE_CLOUD_PROJECT'] = input('Google Cloud project ID: ').strip()
    if not os.environ.get('GOOGLE_CLOUD_PROJECT'):
        raise RuntimeError('GOOGLE_CLOUD_PROJECT is required for planner and gate')
print('Live mode:', not DRY_RUN)
print('Local Gemma endpoint:',JUDGE_BASE_URL)


In [ ]:
# Cell 7: create an isolated runtime configuration
import copy, yaml

RUN_MODE = 'dry' if DRY_RUN else 'live'
resolved_run_id = RUN_ID or f'chembreak_{RUN_MODE}_gemma4_ofs10_seed_{SEED}_6x3'
base_config = PROJECT_ROOT / 'configs' / 'config.yaml'
config = yaml.safe_load(base_config.read_text())
config['run'].update({
    'seed': int(SEED),
    'run_mode': RUN_MODE,
    'run_id': resolved_run_id,
    'dry_run': bool(DRY_RUN),
    'task_limit': TASK_LIMIT,
    'live_progress': True,
    'live_progress_show_text': bool(SHOW_LIVE_TEXT),
    'live_progress_text_limit': int(LIVE_TEXT_LIMIT),
    'prompts_path': str(PROJECT_ROOT / 'data' / 'prompts.csv'),
    'manifest_path': str(PROJECT_ROOT / 'data' / 'prompts_manifest.csv'),
    'lock_path': str(PROJECT_ROOT / 'data' / 'prompts_lock.json'),
    'output_root': str(STORAGE_ROOT / RUN_MODE / resolved_run_id / 'runs'),
    'artifact_root': str(STORAGE_ROOT / RUN_MODE / resolved_run_id / 'task_artifacts'),
})
for role_name in ('attack_planner_llm', 'intent_gate_llm'):
    config['roles'][role_name]['seed'] = int(SEED)
config['roles']['judge_llm']['model'] = GEMMA_JUDGE_MODEL
config['roles']['judge_llm']['base_url'] = JUDGE_BASE_URL
for target in config['targets']:
    target['cache_dir'] = str(STORAGE_ROOT / 'cache' / 'huggingface' / 'hub')
    target['offload_folder'] = str(STORAGE_ROOT / 'offload' / target['id'])

runtime_config_dir = STORAGE_ROOT / RUN_MODE / resolved_run_id
runtime_config_dir.mkdir(parents=True, exist_ok=True)
runtime_config = runtime_config_dir / 'config.runtime.yaml'
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
print('RUN MODE =', RUN_MODE)
print('RUN ID =', resolved_run_id)
print('RUNTIME CONFIG =', runtime_config)


In [ ]:
# Cell 8: package, dataset, and API/GPU preflight
from pprint import pprint
from chembreak.preflight import run_preflight

preflight_report = run_preflight(
    runtime_config,
    probe_tokenizers=False,
    probe_roles=PROBE_ROLES,
    target_id=None,
)
pprint(preflight_report)


## Target 1: ChemDFM

Run the next two cells for ChemDFM. The run cell is independent of the ChemLLM run cell. It unloads ChemDFM in a `finally` block even if execution is interrupted.


In [ ]:
# Cell 10: optional ChemDFM tokenizer preflight
if PROBE_TOKENIZERS:
    pprint(run_preflight(runtime_config, probe_tokenizers=True, probe_roles=False, target_id='ChemDFM'))
else:
    print('ChemDFM tokenizer probe skipped. Set PROBE_TOKENIZERS=True to enable it.')


In [ ]:
# Cell 11: run ChemDFM only
from chembreak.runner import ChemBreakRunner

chemdfm_runner = ChemBreakRunner(runtime_config, 'ChemDFM')
try:
    chemdfm_summary = chemdfm_runner.run_all()
finally:
    chemdfm_runner.unload_target()
    chemdfm_runner.close()

print('ChemDFM FINAL:', chemdfm_summary['bounded_first_success'])


In [ ]:
# Cell 12: confirm GPU memory after ChemDFM
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print('GPU allocated after ChemDFM cleanup (GiB) =', round(torch.cuda.memory_allocated() / 1024**3, 3))
else:
    print('Dry run or no CUDA GPU.')


## Target 2: ChemLLM

Run the next two cells for ChemLLM. This cell creates its own runner and loads only ChemLLM. It does not depend on the ChemDFM runner or its in-memory state.


In [ ]:
# Cell 14: optional ChemLLM tokenizer preflight
if PROBE_TOKENIZERS:
    pprint(run_preflight(runtime_config, probe_tokenizers=True, probe_roles=False, target_id='ChemLLM'))
else:
    print('ChemLLM tokenizer probe skipped. Set PROBE_TOKENIZERS=True to enable it.')


In [ ]:
# Cell 15: run ChemLLM only
from chembreak.runner import ChemBreakRunner

chemllm_runner = ChemBreakRunner(runtime_config, 'ChemLLM')
try:
    chemllm_summary = chemllm_runner.run_all()
finally:
    chemllm_runner.unload_target()
    chemllm_runner.close()

print('ChemLLM FINAL:', chemllm_summary['bounded_first_success'])


In [ ]:
# Cell 16: compare whichever targets have been completed in this session
comparison = {}
if 'chemdfm_summary' in globals():
    comparison['ChemDFM'] = chemdfm_summary['bounded_first_success']
if 'chemllm_summary' in globals():
    comparison['ChemLLM'] = chemllm_summary['bounded_first_success']
if not comparison:
    print('Run at least one target cell first.')
else:
    pprint(comparison)


In [ ]:
# Cell 17: locate public and private outputs
from pathlib import Path

for target_id in ('ChemDFM', 'ChemLLM'):
    target_root = Path(config['run']['output_root']) / config['run']['experiment_revision'] / target_id
    release = target_root / 'release'
    internal = release / 'INTERNAL_DO_NOT_RELEASE'
    print(target_id)
    print('  public release:', release)
    print('  private transcript:', internal / 'chembreak_full_transcripts.csv')
    print('  first-success review:', internal / 'chembreak_first_success_review.csv')


In [ ]:
# Cell 18: create a public-results ZIP without private transcripts
import shutil

zip_root = STORAGE_ROOT / RUN_MODE / resolved_run_id / 'exports'
zip_root.mkdir(parents=True, exist_ok=True)
public_stage = zip_root / 'chembreak_public_results'
if public_stage.exists():
    shutil.rmtree(public_stage)
public_stage.mkdir(parents=True)
for target_id in ('ChemDFM', 'ChemLLM'):
    release = Path(config['run']['output_root']) / config['run']['experiment_revision'] / target_id / 'release'
    if release.exists():
        destination = public_stage / target_id
        shutil.copytree(release, destination, ignore=shutil.ignore_patterns('INTERNAL_DO_NOT_RELEASE'))
public_zip = shutil.make_archive(str(zip_root / f'chembreak_{resolved_run_id}_public'), 'zip', public_stage)
print('PUBLIC ZIP =', public_zip)


In [ ]:
# Cell 19: create a private audit ZIP containing raw prompts and responses
import shutil

private_stage = zip_root / 'chembreak_private_audit'
if private_stage.exists():
    shutil.rmtree(private_stage)
private_stage.mkdir(parents=True)
for target_id in ('ChemDFM', 'ChemLLM'):
    internal = Path(config['run']['output_root']) / config['run']['experiment_revision'] / target_id / 'release' / 'INTERNAL_DO_NOT_RELEASE'
    if internal.exists():
        shutil.copytree(internal, private_stage / target_id)
private_zip = shutil.make_archive(str(zip_root / f'chembreak_{resolved_run_id}_private_audit'), 'zip', private_stage)
print('PRIVATE AUDIT ZIP =', private_zip)
